# LouisFarm — Semaine 4 : Data Visualization & Storytelling
## Dataset : Inclusion Financiere Cote d Ivoire

**Objectif :** Transformer l analyse en communication decisionnelle visuelle et narrative.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import seaborn as sns; sns.set_theme(style="whitegrid")
from scipy import stats
import warnings; warnings.filterwarnings("ignore")

np.random.seed(42)
n = 800
regions = ["Abidjan","Bouake","Daloa","Korhogo","San-Pedro","Man","Yamoussoukro"]
df = pd.DataFrame({
    "region": np.random.choice(regions, n),
    "zone": np.random.choice(["Urbain","Periurbain","Rural"], n, p=[0.45,0.3,0.25]),
    "genre": np.random.choice(["Homme","Femme"], n),
    "taux_bancarisation": np.random.beta(2.5, 4, n)*100,
    "taux_mobilemoney": np.random.beta(3, 2, n)*100,
    "revenu_mensuel_fcfa": np.random.lognormal(11.5, 0.8, n),
    "distance_agence_km": np.random.exponential(12, n),
    "niveau_education": np.random.choice(["Aucun","Primaire","Secondaire","Superieur"], n, p=[0.2,0.35,0.3,0.15]),
    "utilisation_credit": np.random.choice([0,1], n, p=[0.65,0.35]),
    "acces_internet": np.random.choice([0,1], n, p=[0.4,0.6]),
})
df.loc[df.zone=="Rural","taux_bancarisation"] *= 0.5
df.loc[df.zone=="Rural","taux_mobilemoney"] *= 0.8
df.loc[df.zone=="Urbain","taux_bancarisation"] *= 1.5
df["taux_bancarisation"] = df.taux_bancarisation.clip(2,95)
df["taux_mobilemoney"] = df.taux_mobilemoney.clip(5,98)
print(f"Dataset: {df.shape}")

## Lecon 4.1 — Matplotlib : 6 visualisations analytiques

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle("Analytical Storyboard — Inclusion Financiere CI 2023", fontsize=14, fontweight="bold")

# 1. Bancarisation par zone
by_zone = df.groupby("zone")[["taux_bancarisation","taux_mobilemoney"]].mean()
x = np.arange(len(by_zone)); w = 0.35
axes[0,0].bar(x-w/2, by_zone.taux_bancarisation, w, label="Bancarisation", color="#2E86AB")
axes[0,0].bar(x+w/2, by_zone.taux_mobilemoney, w, label="Mobile Money", color="#F18F01")
axes[0,0].set_xticks(x); axes[0,0].set_xticklabels(by_zone.index)
axes[0,0].set_title("Services financiers par zone"); axes[0,0].legend()

# 2. Distribution taux bancarisation
for zone, col in zip(["Urbain","Periurbain","Rural"],["#2E86AB","#F18F01","#C73E1D"]):
    axes[0,1].hist(df[df.zone==zone].taux_bancarisation, bins=20, alpha=0.6, color=col, label=zone, density=True)
axes[0,1].set_title("Distribution par zone"); axes[0,1].legend()

# 3. Genre vs credit
ct = pd.crosstab(df.genre, df.utilisation_credit, normalize="index")*100
ct.plot(kind="bar", ax=axes[0,2], color=["#E8E8E8","#2E86AB"])
axes[0,2].set_title("Utilisation du credit par genre"); plt.setp(axes[0,2].get_xticklabels(), rotation=0)

# 4. Scatter revenu vs mobile money
for zone, col in zip(["Urbain","Periurbain","Rural"],["#2E86AB","#F18F01","#C73E1D"]):
    sub = df[df.zone==zone]
    axes[1,0].scatter(sub.revenu_mensuel_fcfa/1000, sub.taux_mobilemoney, alpha=0.4, s=20, color=col, label=zone)
axes[1,0].set_xlabel("Revenu (milliers FCFA)"); axes[1,0].set_title("Revenu vs Mobile Money")
axes[1,0].legend(); axes[1,0].set_xlim(0,500)

# 5. Heatmap correlations
corr_cols = ["taux_bancarisation","taux_mobilemoney","revenu_mensuel_fcfa","distance_agence_km","utilisation_credit","acces_internet"]
sns.heatmap(df[corr_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1,1])
axes[1,1].set_title("Matrice de correlation")

# 6. Education vs bancarisation
edu = ["Aucun","Primaire","Secondaire","Superieur"]
edu_data = df.groupby("niveau_education")["taux_bancarisation"].mean().reindex(edu)
axes[1,2].bar(range(4), edu_data.values, color=["#C73E1D","#F18F01","#2E86AB","#3B1F2B"])
axes[1,2].set_xticks(range(4)); axes[1,2].set_xticklabels(edu, rotation=20, ha="right")
axes[1,2].set_title("Education et bancarisation")

plt.tight_layout()
plt.savefig("./s4_storyboard.png", dpi=120, bbox_inches="tight")
plt.show()
print("Storyboard sauvegarde")

## Lecon 4.2 — Executive Summary

In [ ]:
city_bank = df[df.zone=="Urbain"].taux_bancarisation.mean()
rural_bank = df[df.zone=="Rural"].taux_bancarisation.mean()
mm_int = df[df.acces_internet==1].taux_mobilemoney.mean()
mm_no = df[df.acces_internet==0].taux_mobilemoney.mean()
fem_cred = df[df.genre=="Femme"].utilisation_credit.mean()*100
hom_cred = df[df.genre=="Homme"].utilisation_credit.mean()*100

print("EXECUTIVE SUMMARY — Inclusion Financiere CI 2023")
print("=" * 60)
print(f"CONSTAT: Ecart urbain-rural bancarisation = {city_bank-rural_bank:.0f} pts")
print(f"INSIGHT: Internet multiplie adoption mobile money par {mm_int/mm_no:.1f}x")
print(f"ALERTE : Femmes {hom_cred-fem_cred:.0f}% moins utilisatrices du credit")
print(f"RECOMMANDATION: Investir en infrastructure internet rurale")

## Exercice 4.1 — Distance agence et bancarisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df["dist_bin"] = pd.cut(df.distance_agence_km, bins=[0,5,15,30,100],
                         labels=["<5km","5-15km","15-30km",">30km"])
by_dist = df.groupby("dist_bin", observed=True).taux_bancarisation.mean()
axes[0].bar(range(4), by_dist.values, color=["#2E86AB","#5BA4CF","#F18F01","#C73E1D"])
axes[0].set_xticks(range(4)); axes[0].set_xticklabels(by_dist.index)
axes[0].set_title("Bancarisation par distance a l agence")

x = df.distance_agence_km.clip(0,60)
y = df.taux_bancarisation
sl, ic, r, p, _ = stats.linregress(x, y)
axes[1].scatter(x, y, alpha=0.3, s=15, color="#2E86AB")
axes[1].plot([0,60],[ic,ic+sl*60],"r-",lw=2)
axes[1].set_title(f"Scatter (r={r:.2f})")
axes[1].text(0.5,0.95,f"Chaque km => {sl:.2f}% bancarisation en moins",
             transform=axes[1].transAxes, ha="center", va="top", fontsize=9,
             bbox=dict(boxstyle="round",facecolor="wheat",alpha=0.7))

plt.tight_layout()
plt.savefig("./s4_exercice.png", dpi=100, bbox_inches="tight")
plt.show()